# 08 — Listing Lens V4: AI Generator Adaptation and Fine-Tuning

> **Notebook status: documented run record plus reusable training entry points.** This notebook records the completed V4 experiment. The saved V4 model and run report are included. The raw V2 images, new AI candidate images and historical V3 starting checkpoint are not bundled, so a complete training rerun requires those inputs to be supplied separately.

V4 fine-tunes the earlier EfficientNetB0 V3 model. It does not train a new CNN from random initialisation. The final application ships only the V4 weights.


## 1. V4 dataset construction

Dataset V2 was copied without changing the original real-image folders. Fifty-four new AI-labelled candidates were added: GPT 26, Gemini 19 and Meta AI 9. Exact duplicates skipped: 0. Cross-dataset perceptual near-duplicate pairs flagged: 0. All 1,304 resulting files were loaded by the V4 training pipeline.

| Split | Real | AI-generated | Total |
|---|---:|---:|---:|
| Training | 437 | 474 | 911 |
| Validation | 94 | 102 | 196 |
| Testing | 94 | 103 | 197 |
| **Total** | **625** | **679** | **1,304** |


In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
REPORT = ROOT / 'documentation' / 'v4_training_report.json'
report = json.loads(REPORT.read_text(encoding='utf-8'))
print(json.dumps(report['dataset_counts'], indent=2))
print('Threshold:', report['selected_threshold_from_validation_balanced_accuracy'])
print('V4 SHA-256:', report['v4_candidate_sha256'])


## 2. Recorded training configuration

| Setting | Phase 1 | Phase 2 |
|---|---|---|
| Backbone | Frozen | Top 40 layers fine-tuned; Batch Normalization frozen |
| Learning rate | 0.0001 | 0.00001 |
| Maximum epochs | 5 | 12 |
| Actual recorded Phase 2 epochs | — | 8 |

Additional settings: Adam optimizer, binary cross-entropy with label smoothing 0.05, batch size 8, class weights, random seed 42, early stopping and learning-rate reduction. Input was RGB 256 × 256 with raw float32 pixel values 0–255; model-internal EfficientNetB0 preprocessing was retained.


## 3. Re-running (requires external inputs)

The scripts are in `training/`. To reconstruct the run, provide (a) the Dataset V2 source split, (b) the 54 original candidate images plus their candidate manifest, and (c) the historical V3 checkpoint whose SHA-256 is `ea47dc1e9ded13944a2ac866ae968f3956a003d9faaf39ce4a0dfa8a8cad65b5`. V3 weights are intentionally not bundled because the final application uses V4 only. The scripts create new `data_v4/` and `outputs_v4/` folders and never overwrite the active model.


In [ ]:
from pathlib import Path
import subprocess, sys

dataset_root = ROOT / 'data_v4'
v3_checkpoint = ROOT / 'external_checkpoints' / 'car_cnn_model_v3.keras'
if not dataset_root.is_dir():
    print('Dataset not found:', dataset_root, '— run training/prepare_v4_dataset.py after supplying the source images.')
elif not v3_checkpoint.is_file():
    print('Historical V3 starting checkpoint not supplied. The V4 training run is documented but cannot be repeated from this repository alone.')
else:
    command = [sys.executable, 'training/train_v4.py', '--dataset-root', str(dataset_root), '--v3-model', str(v3_checkpoint), '--output-dir', 'outputs_v4']
    print('Run command:', ' '.join(command))
    print('Review the training inputs and launch the command in Terminal when ready.')
